# Neural Network project
***

In this Jupyter Notebook, we will learn to create and fit a Neural Network.

* First, we will use simulated data of a hospital. It contains 8 variables and one simulated outcome "cardio metabolic risk". We will learn how to fit a model using it.

* Second, you will get to use the syntax defined to fit your own network on a new data set.



## *1. Introduction to Neural Networks*

We will first import the data we cleaned last week. We had the folowing columns:

As Explanatory variables (Features):
```
8x medical variables...
```

As Dependant variable (Label):

```
cardio_metabolic_risk : Binary variable stating risk for cardio metabolic disease
```

**The objective of our Neural network will be to predict the cardio metabolic risk of a patient based on the explanatory variables.**



In [ ]:
#importing python libraries needed for this Notebook
import numpy as np
import pandas as pd
from pathlib import Path

#Setting options
pd.set_option('display.max_columns', None)


# Reading data into df
DATA_PATH = Path('cleaned_dat.csv')
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# Seperating features and label
feature_cols = df.columns[0:8]
label_col = "cardio_metabolic_risk"

X = df[feature_cols].values
y = df[label_col].values

## *1.1 Neural Network Initiation*

A neural network consists of 2 simple components organised in layers:

* **Neurons**, which are the nodes in neural networks. They have both an **aggregation** and an **activation** function, these function define how the incoming information into the neurons is processed and sent further. These function are "hyperparameters", they can be freely selected when designing a Network.

* **Weights**, which are the links between the neurons

These components are organised in 3 types of layers:

* **Input layer**: The first and unique layer in which the data is inputed into the network. (Each neuron  in the input layer is a column of the data)

* **Output layer**: The last and unique layer in which the result of the model can be read from.

* **Hidden layers**: The many layers between the input and output layers




| <img src= "https://upload.wikimedia.org/wikipedia/commons/d/d2/Neural_network_explain.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail_unscaled&_=20211111130354" alt="Neural Network" width="800"> |
|:--:|
|Figure 1 - Simple representation of a Neural network with an input layer, two hidden layers and an Output layer.
source: Wikimedia Commons ([link](https://commons.wikimedia.org/wiki/File:Neural_network_explain.png))|


### *PyTorch*
There are many packages that can be used to create Neural Networks in Python. The most popular are *PyTorch* and *TensorFlow*. In this notebook we will be using *PyTorch*. It has many different implementations, please use the correct comand in Pip to install it. You can look for the command here: [PyTorch website](https://pytorch.org/get-started/locally/).

In PyTorch, Neural Network are defined as classes. We treat them as objects we can initiate, call and modify. If you are unfamiliar with object oriented programm please go through this [quick guide](https://www.geeksforgeeks.org/python/python-object/). It will make understanding PyTorch syntax a lot easier.

### *Defining the network:*

We will define a model with the following properties:

* One input layer containing all our input columns

* Three hidden layers containing 10, 20 and 10 nodes respectively

* One output layer containing only one node, the probablity of the cardio metabolic risk being present.


At the same time we define the Aggregation and activation functions:

```
nn.Linear() # Defines the aggregation function as linear
nn.Relu()   # Defines the activation function as Relu
```





In [ ]:
import torch
import torch.nn as nn # Importing Pytorch neural networks
import torch.optim as optim # Importing Pytorch optimizer

# Defining our neural network object
class simpleNN(nn.Module):
    def __init__(self):

        super().__init__()
        self.layer_1 = nn.Linear(in_features=8, out_features=10) # Defining input + first hidden layer
        self.layer_2 = nn.Linear(in_features=10, out_features=20) # Defining second hidden layer
        self.layer_3 = nn.Linear(in_features=20, out_features=10) # Defining third hidden layer
        self.layer_4 = nn.Linear(in_features=10, out_features=1) # Defining output layer

        self.relu = nn.ReLU() # Defining an activation function.

    def forward(self, x):
      # Interspersing the ReLU activation function between layers
       return self.layer_4(self.relu(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))))

In [ ]:
# Saving an instance of the class in the "model" object.
my_model = simpleNN()

#printing the model
print(my_model)

## *1.2 Neural Network predictions and learning*

To make a prediction, neural networks take as input an observation of a dataset and propagates it through the nodes, using the weights, aggregation functions and activation functions. The value of the activation function of the output layer, in our case a single node, is the prediction made by the model. This whole process is called ***forward propagation***.

Right now, the model we defined will still make decisions completely at random. This is because, despite defining the network, the weights are still arbitrary. In order to get accurate predictions, the model needs to "learn" the exact weights to make precise predictions. These weights are learned from the training data in a step called ***backwards propagation***. This consists in slightly modulating the weights in order to get more accurate predictions. This modulation is not done at random, but done in a way to minimize a ***loss function***. The loss function describes how far the prediction was from the true value, and also is a Hyperparameter.

The whole process of forward and backward propagation is repeated many times, until the weights are properly defined. Each repetition is called an ***epoch***.


In [ ]:
#Transforming the data in to a tensor so that PyTorch can use it
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)


# Initiate a loss function
loss_fn = nn.BCEWithLogitsLoss()

# Initiate an optimizer with a given learning rate
optimizer = torch.optim.SGD(params=my_model.parameters(), lr=0.0001)


#  Create a performance metric for us to understand what is going on - in this case, proportion of correctly predicted values
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc = (correct / len(y_pred)) * 100
    return acc


### *1.3 Overfitting*

Now that we have been able to define our network, we would like to find a strategy to correctly assess its performance. For the moment, we implemented an "accuracy" measure to have an idea of how the model is training, however assessing model performance based on this is a mistake.

Indeed, we are trying to have this model predict the performance of real life data and not our specific dataset only. By testing the model on the learning data, the model just repeats the labels of the samples it has just seen 3000 times. This results in a perfect score on the training data but would fail to predict anything useful on unseen data. This situation is called ***overfitting***. In contrast, ***underfitting*** data using a Neural Network is also possible, this would happen if we did not create enough hidden layers and the complexity of the model would be too low for the data.

|<img src='https://scikit-learn.org/stable/_images/sphx_glr_plot_underfitting_overfitting_001.png' title='classifiers' width='80%'>|
|:--:|
|Figure 2 - Two dimensional example demonstrating the problems of underfitting (left) and overfitting (right).
source: [scikit-learn](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html) |



### *1.4 Train/test split*

In order to avoid overfitting the Neural Network, we do not test it on the same data it has been trained on. Unfortunately, we only have one dataset available. To solve this issue, we have to split the data before training the model into two datasets:

* ***A training set:*** Containing most of the data points, will be used to train the model
* ***A test set:***     Containing a small part of the data points that our model has never seen.


| <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Machine_learning_nutshell_--_Split_into_train-test_set.svg/1280px-Machine_learning_nutshell_--_Split_into_train-test_set.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20160224183504" alt="image of splitted data" width="800"> |
|:--:|
|Figure 1 - Simple representation of data set splitting
source: Wikimedia commons|

In [ ]:
from sklearn.model_selection import train_test_split

#Splitting the data with 80% going to the training set and 20% the test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### *1.5 Fitting the model*

For a reminder of the steps involved: see section 1.2.

In [ ]:
#########################################
###     Fitting the Neural Network    ###
#########################################

torch.manual_seed(42)

#Defining the number of epochs
epochs = 50000

# Set data to be processed by CPU
X_train, y_train = X_train.to("cpu"), y_train.to("cpu")
X_test, y_test = X_test.to("cpu"), y_test.to("cpu")


for epoch in range(epochs):
    # 1. Forward pass
    y_logits = my_model(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits))

    # 2. Calculate loss and accuracy on training data
    loss = loss_fn(y_logits, y_train)
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    # 3. Resets optimizer
    optimizer.zero_grad()

    # 4. Calculates loss function through backwards propagation
    loss.backward()

    # 5. Adjusts the weight
    optimizer.step()

    # 6. Evaluating model
    my_model.eval()

    # 7. Test set evaluation
    with torch.inference_mode():
      # 1. Forward pass
      test_logits = my_model(X_test).squeeze()
      test_pred = torch.round(torch.sigmoid(test_logits)) # logits -> prediction probabilities -> prediction labels

      # 2. Calculate loss and accuracy
      test_loss = loss_fn(test_logits, y_test)
      test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

    # Print out what's happening
    if epoch % 2000 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Accuracy: {test_acc:.2f}%")



### *1.6 Result*

We made a model that has a prediction accuracy of over 90% !!!

## *2. Exercise - Predicting strokes at the General Hospital*

You receive another dataset from the General Hospital (GH). It is built as follows:

Features:
> bmi : Body mass index of patient (Z-scored)

> smoke : Binary variable containing 1 if the patient smokes

> rest_bpm : Resting heart rate of the patient (Z-scored)

Label:
> Stroke : Variable indicating if the patient had a stroke


Build a Neural network to predict if a patient is at risk of stroke or not using their bmi, smoking status and resting heart rate. Your Network should have the following properties:


* The model should have at least 3 hidden layers

* The model cannot have more than 15 nodes per layer

* The test accuracy should be higher than 70%


***Tip: You can keep the learning rate, aggregation and activation functions from above but you may need to increase the number of epochs***

**Use of AI declaration:** This Notebook, including all code and text it contains, was produced without support from Artificial Inteligence (AI) tools.